# BitcoinGraphGuard — Phase 3: GraphSAGE Baseline

**Phase:** 3 of 9 · Homogeneous Graph Neural Network Baseline
**Environment:** Google Colab / Kaggle. This notebook is **never executed on the laptop**.
**Input:** The real Elliptic++ dataset in `Og data/` (`txs_features.csv`, `txs_classes.csv`, `txs_edgelist.csv`), using the protocol and findings from `docs/EDA.md` and `docs/XGBOOST.md`.

Model progression for the project:

```text
Simple baseline -> Logistic Regression -> XGBoost baseline -> GraphSAGE baseline -> RGCN / HGT
```

This notebook implements the **homogeneous Graph Neural Network baseline (GraphSAGE)** and its paired feature-only neural ablation (**MLP**). It answers the central research question:

> **"Can graph neighborhood information improve illicit transaction detection compared with the frozen transaction-level XGBoost baseline?"**

### Frozen Reference Numbers from Phase 2 (`docs/XGBOOST.md`):
* **XGBoost Optimized (165 features):** Test 35–49 PR-AUC: **0.8013**, ROC-AUC: **0.9281**, F1: **0.7818**
* **XGBoost Early Window (35–42):** PR-AUC: **0.9215** (prevalence 9.16%)
* **XGBoost Drift Window (43–49):** PR-AUC: **0.0427** (prevalence 2.53%)
* **Logistic Regression:** Test 35–49 PR-AUC: **0.2917**, ROC-AUC: **0.8828**
* **Prevalence Baseline:** Test 35–49 PR-AUC: **0.0650**, ROC-AUC: **0.5000**

### Key Methodological Principles in this Notebook:
1. **Homogeneous Transaction Graph:** 203,769 nodes, 234,355 directed transaction-to-transaction edges.
2. **Empirical Temporal Graph Locality Verification:** Verifies the EDA structural finding that 100% of transaction edges are strictly intra-step (zero cross-step edges in `txs_edgelist.csv`).
3. **Strict Temporal Protocol:** Fit window (1–24) -> Validation (25–34) -> Refit window (1–34) -> Primary Test (35–49) -> Sub-windows (35–42 & 43–49).
4. **Zero Data Leakage:** Feature scaling (`StandardScaler`) and loss weighting (`pos_weight`) are fit exclusively on historical training data.
5. **Controlled Ablation:** 2-layer GraphSAGE vs identical 2-layer MLP (isolating graph aggregation from neural architecture).
6. **Operating Point Selection:** F1-maximization on validation window (25–34) only, frozen before test evaluation.

**To run:** `Runtime -> Run all`. Everything is written to `<Drive>/Projects/Bitcoin Graph/graphsage/`.

## 1. Setup & Environment

One config cell. Configures random seeds, device detection (CUDA / CPU), time-step boundaries, directory resolution, and the validation check recording system (`CHECKS`). No personal paths are hardcoded.

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from IPython.display import display
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from torch import nn

# Try importing PyG; provide transparent notification if not present
try:
    from torch_geometric.nn import SAGEConv

    PYG_AVAILABLE = True
except ImportError:
    PYG_AVAILABLE = False
    print(
        "PyG (torch_geometric) not detected in environment. Using self-contained PyTorch GraphSAGE implementation."
    )

SEED = 42
N_STEPS = 49  # Elliptic++ covers 49 contiguous time steps

# Temporal protocol windows (established in EDA and Phase 2)
FIT_MIN, FIT_MAX = 1, 24
VAL_MIN, VAL_MAX = 25, 34
TRAIN_MIN, TRAIN_MAX = 1, 34
TEST_MIN, TEST_MAX = 35, 49
TEST_EARLY_MAX = 42  # 35-42 sub-window
DRIFT_MIN = 43  # 43-49 secondary recent-drift window

LABEL_NAMES = {1: "illicit", 2: "licit", 3: "unknown"}

# Deterministic seeding for exact reproducibility
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

sns.set_theme(context="notebook", style="whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 11

print(
    f"python {sys.version.split()[0]} | numpy {np.__version__} | pandas {pd.__version__} | torch {torch.__version__} | device {device}"
)

In [ ]:
FILES = {
    "txs_features": "txs_features.csv",
    "txs_classes": "txs_classes.csv",
    "txs_edgelist": "txs_edgelist.csv",
    "addr_tx": "AddrTx_edgelist.csv",
    "tx_addr": "TxAddr_edgelist.csv",
}

# One Drive folder holds the raw dataset and every notebook's outputs.
DRIVE_ROOT = Path(
    os.environ.get(
        "BITCOINGUARD_DRIVE_ROOT", "/content/drive/MyDrive/Projects/Bitcoin Graph"
    )
)


def resolve_data_dir() -> Path:
    """Find the Elliptic++ CSVs in Colab, Kaggle or a local checkout."""
    override = os.environ.get("ELLIPTIC_DATA_DIR")
    candidates = [
        Path(override) if override else None,
        DRIVE_ROOT / "Dataset",
        Path("/content/drive/MyDrive/elliptic-plus-plus"),
        Path("/content/elliptic-plus-plus"),
        Path("/kaggle/input/elliptic-plus-plus"),
        Path("/kaggle/input/elliptic"),
        Path.cwd() / "Og data",
    ]
    for candidate in candidates:
        if (
            candidate
            and candidate.is_dir()
            and (candidate / FILES["txs_features"]).exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Elliptic++ not found; set ELLIPTIC_DATA_DIR to the folder with the CSVs."
    )


def read_header(path: Path) -> list:
    """Column names of a CSV without reading data rows."""
    return pd.read_csv(path, nrows=0).columns.tolist()


DATA_DIR = resolve_data_dir()
OUT_DIR = DRIVE_ROOT / "graphsage"
FIG_DIR = OUT_DIR / "figures"
for directory in (OUT_DIR, FIG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

CHECKS: list = []


def check(name: str, value, expected=None, note: str = "") -> None:
    """Record a sanity check for section 13 validation reporting."""
    ok = expected is None or value == expected
    CHECKS.append(
        {
            "check": name,
            "value": value,
            "expected": expected,
            "ok": bool(ok),
            "note": note,
        }
    )
    flag = "ok" if ok else "FAIL"
    print(
        f"[{flag}] {name}: {value}"
        + (f" (expected {expected})" if expected is not None else "")
    )


def save_table(name: str, frame: pd.DataFrame) -> None:
    frame.to_csv(OUT_DIR / name, index=False)


def save_json(name: str, payload: dict) -> None:
    (OUT_DIR / name).write_text(
        json.dumps(payload, indent=2, default=str), encoding="utf-8"
    )


def save_fig(name: str) -> None:
    plt.tight_layout()
    plt.savefig(FIG_DIR / name, dpi=130, bbox_inches="tight")
    plt.show()


MODELS: dict = {}


def register_model(key: str, label: str, model_obj, scores_dict: dict) -> None:
    """Record a model and its prediction outputs for unified evaluation."""
    MODELS[key] = {"label": label, "model": model_obj, "scores": scores_dict}


print(f"data : {DATA_DIR}")
print(f"out  : {OUT_DIR}")

## 2. Load Data

The raw dataset files are opened read-only and never modified.

We load:
1. `txs_features.csv`: 165 primary transaction features (93 `Local_feature_*` + 72 `Aggregate_feature_*`). The 17 domain columns are excluded at parse time due to near-perfect collinearity ($|r| \approx 0.98 - 1.00$) and 16,405 missing values.
2. `txs_classes.csv`: Transaction labels (1 = illicit, 2 = licit, 3 = unknown).
3. `txs_edgelist.csv`: Transaction-to-transaction directed edges (`txId1 -> txId2`).

`has_addresses` is excluded following the Phase 2 ablation finding that it does not provide positive discriminative signal (all 4,545 illicit transactions have address links).

In [ ]:
txs_columns = read_header(DATA_DIR / FILES["txs_features"])
all_feature_columns = [c for c in txs_columns if c not in ("txId", "Time step")]
domain_columns = [
    c
    for c in all_feature_columns
    if not c.startswith(("Local_feature", "Aggregate_feature"))
]
model_features = [c for c in all_feature_columns if c not in domain_columns]

print(
    f"{len(all_feature_columns)} total features = {len(domain_columns)} domain (excluded) + {len(model_features)} model features"
)

check("features: total raw features", len(all_feature_columns), 182)
check("features: domain columns excluded", len(domain_columns), 17)
check("features: primary model features", len(model_features), 165)
check(
    "features: domain block is disjoint from model features",
    set(domain_columns).isdisjoint(model_features),
    True,
)

txs_classes = pd.read_csv(DATA_DIR / FILES["txs_classes"], dtype={"txId": str})
check("labels: transaction rows in classes CSV", len(txs_classes), 203_769)
check(
    "labels: classes are subset of {1, 2, 3}",
    set(txs_classes["class"].unique()) <= {1, 2, 3},
    True,
)

In [ ]:
started = time.time()
txs = pd.read_csv(
    DATA_DIR / FILES["txs_features"],
    usecols=["txId", "Time step"] + model_features,
    dtype={"txId": str, "Time step": "int16", **{c: "float32" for c in model_features}},
)
txs["class"] = txs["txId"].map(txs_classes.set_index("txId")["class"]).astype("int8")

print(
    f"Loaded {len(txs):,} transactions in {time.time() - started:.1f}s | Memory: {txs.memory_usage(deep=True).sum() / 1e6:.1f} MB"
)

check("transactions: total row count", len(txs), 203_769)
check("transactions: zero missing labels", int(txs["class"].isna().sum()), 0)
check(
    "transactions: time step span",
    (int(txs["Time step"].min()), int(txs["Time step"].max())),
    (1, 49),
)
check(
    "transactions: missing values in 165 model features",
    int(txs[model_features].isna().to_numpy().sum()),
    0,
)

label_counts = {
    LABEL_NAMES[int(k)]: int(v) for k, v in txs["class"].value_counts().items()
}
print("Transaction label distribution:", label_counts)
check("transactions: illicit count", label_counts.get("illicit"), 4_545)
check("transactions: licit count", label_counts.get("licit"), 42_019)
check("transactions: unknown count", label_counts.get("unknown"), 157_205)

In [ ]:
# Map string txId to contiguous integer index [0, N-1]
tx_id_to_idx = {tx_id: idx for idx, tx_id in enumerate(txs["txId"].values)}
idx_to_tx_id = {idx: tx_id for tx_id, idx in tx_id_to_idx.items()}

# Load transaction edgelist
txs_edges_df = pd.read_csv(DATA_DIR / FILES["txs_edgelist"], dtype=str)
print(
    f"Loaded {len(txs_edges_df):,} raw transaction edges from {FILES['txs_edgelist']}"
)

check("graph: raw transaction edges count", len(txs_edges_df), 234_355)
check(
    "graph: edge source nodes in node universe",
    bool(txs_edges_df["txId1"].isin(tx_id_to_idx).all()),
    True,
)
check(
    "graph: edge target nodes in node universe",
    bool(txs_edges_df["txId2"].isin(tx_id_to_idx).all()),
    True,
)

# Map edge endpoints to node indices
src_indices = txs_edges_df["txId1"].map(tx_id_to_idx).values.astype(np.int64)
dst_indices = txs_edges_df["txId2"].map(tx_id_to_idx).values.astype(np.int64)

# Create PyTorch edge_index tensor [2, E]
edge_index_np = np.stack([src_indices, dst_indices], axis=0)
edge_index = torch.tensor(edge_index_np, dtype=torch.long)

print(
    f"Constructed edge_index tensor: shape {edge_index.shape}, dtype {edge_index.dtype}"
)
check("graph: edge_index shape", tuple(edge_index.shape), (2, 234_355))

## 3. Graph Construction & Intra-Step Temporal Diagnostics

### Structural Constraint of `txs_edgelist`
A critical architectural property established in the EDA (`docs/EDA.md`) is that **every transaction-to-transaction edge occurs entirely within a single time step**. That is, for all edges $(u, v) \in E_{\text{tx}}$, $\text{TimeStep}(u) = \text{TimeStep}(v)$.

Below, we empirically verify this property across all 234,355 edges. This establishes why homogeneous GraphSAGE operates purely as **intra-step message passing** and cannot propagate cross-time-step signals without wallet/address nodes.

In [ ]:
node_steps = txs["Time step"].values

src_steps = node_steps[src_indices]
dst_steps = node_steps[dst_indices]
cross_step_edges = int((src_steps != dst_steps).sum())
same_step_edges = int((src_steps == dst_steps).sum())

print(f"Total edges: {len(src_indices):,}")
print(
    f"Intra-step edges (src step == dst step): {same_step_edges:,} ({100 * same_step_edges / len(src_indices):.2f}%)"
)
print(
    f"Cross-step edges (src step != dst step): {cross_step_edges:,} ({100 * cross_step_edges / len(src_indices):.2f}%)"
)

check("graph: 100% of transaction edges are intra-step", same_step_edges, 234_355)
check(
    "graph: zero cross-step transaction edges",
    cross_step_edges,
    0,
    note="Proves transaction-only GNN message passing is strictly confined within each time step",
)

# Compute Degree Distributions
in_degrees = np.bincount(dst_indices, minlength=len(txs))
out_degrees = np.bincount(src_indices, minlength=len(txs))
total_degrees = in_degrees + out_degrees

txs["in_degree"] = in_degree = in_degrees
txs["out_degree"] = out_degree = out_degrees
txs["total_degree"] = total_degree = total_degrees

isolated_nodes = int((total_degrees == 0).sum())
connected_nodes = int((total_degrees > 0).sum())
print(
    f"Node degree summary: isolated nodes = {isolated_nodes:,} ({100 * isolated_nodes / len(txs):.2f}%), "
    f"connected nodes = {connected_nodes:,} ({100 * connected_nodes / len(txs):.2f}%)"
)
print(
    f"Max in-degree: {in_degrees.max()}, Max out-degree: {out_degrees.max()}, Max total degree: {total_degrees.max()}"
)

check("graph: isolated transaction nodes count", isolated_nodes, 143_084)
check("graph: connected transaction nodes count", connected_nodes, 60_685)

graph_diag = pd.DataFrame(
    [
        {"metric": "total_nodes", "value": len(txs)},
        {"metric": "total_edges", "value": len(src_indices)},
        {"metric": "intra_step_edges", "value": same_step_edges},
        {"metric": "cross_step_edges", "value": cross_step_edges},
        {"metric": "isolated_nodes", "value": isolated_nodes},
        {"metric": "connected_nodes", "value": connected_nodes},
        {"metric": "max_in_degree", "value": int(in_degrees.max())},
        {"metric": "max_out_degree", "value": int(out_degrees.max())},
        {
            "metric": "mean_degree_connected",
            "value": float(total_degrees[total_degrees > 0].mean()),
        },
    ]
)
save_table("graph_diagnostics.csv", graph_diag)
display(graph_diag)

## 4. Define Temporal Protocol & Split Masks

We maintain the exact, leakage-free temporal split protocol adopted in Phase 1 (EDA) and Phase 2 (XGBoost):

| Split Window | Time Steps | Role in Pipeline | Labeled Rows | Illicit Rows | Illicit Share |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **`fit`** | 1–24 | Model selection, training, early stopping validation base | 23,606 | 2,219 | 9.40% |
| **`validation`** | 25–34 | Validation PR-AUC early stopping & threshold selection | 6,288 | 1,243 | 19.77% |
| **`train`** | 1–34 | Full historical refit before test deployment | 29,894 | 3,462 | 11.58% |
| **`test`** | 35–49 | Primary frozen test benchmark | 16,670 | 1,083 | 6.50% |
| **`test_early`** | 35–42 | High-prevalence early test sub-window | 9,983 | 914 | 9.16% |
| **`test_drift`** | 43–49 | Low-prevalence secondary recent-drift sub-window | 6,687 | 169 | 2.53% |

**Rules:**
1. Only labeled transactions (classes 1 and 2) are included in evaluation metrics and supervised loss.
2. Class 3 (unknown) is never treated as licit.
3. Test sets (35–49) are never touched during scaling, training, or threshold selection.

In [ ]:
step_series = txs["Time step"]
is_labeled_series = txs["class"].isin([1, 2])

mask_fit = (
    (step_series >= FIT_MIN) & (step_series <= FIT_MAX) & is_labeled_series
).values
mask_val = (
    (step_series >= VAL_MIN) & (step_series <= VAL_MAX) & is_labeled_series
).values
mask_train = (
    (step_series >= TRAIN_MIN) & (step_series <= TRAIN_MAX) & is_labeled_series
).values
mask_test = (
    (step_series >= TEST_MIN) & (step_series <= TEST_MAX) & is_labeled_series
).values
mask_test_early = (
    (step_series >= TEST_MIN) & (step_series <= TEST_EARLY_MAX) & is_labeled_series
).values
mask_test_drift = (
    (step_series >= DRIFT_MIN) & (step_series <= TEST_MAX) & is_labeled_series
).values

split_masks = {
    "fit": mask_fit,
    "validation": mask_val,
    "train": mask_train,
    "test": mask_test,
    "test_early": mask_test_early,
    "test_drift": mask_test_drift,
}

window_rows = []
for name, mask in split_masks.items():
    labels = txs.loc[mask, "class"]
    illicit_cnt = int((labels == 1).sum())
    licit_cnt = int((labels == 2).sum())
    tot = illicit_cnt + licit_cnt
    window_rows.append(
        {
            "window": name,
            "steps": {
                "fit": "1-24",
                "validation": "25-34",
                "train": "1-34",
                "test": "35-49",
                "test_early": "35-42",
                "test_drift": "43-49",
            }[name],
            "labeled": tot,
            "illicit": illicit_cnt,
            "licit": licit_cnt,
            "illicit_pct": round(100 * illicit_cnt / max(1, tot), 2),
        }
    )
window_table = pd.DataFrame(window_rows)
save_table("split_windows.csv", window_table)
display(window_table)

check("protocol: zero train/test overlap", bool((mask_train & mask_test).any()), False)
check(
    "protocol: sub-windows partition test period",
    int(mask_test_early.sum() + mask_test_drift.sum()),
    int(mask_test.sum()),
)
check("protocol: fit labeled count", int(mask_fit.sum()), 23_606)
check("protocol: validation labeled count", int(mask_val.sum()), 6_288)
check("protocol: train labeled count", int(mask_train.sum()), 29_894)
check("protocol: test labeled count", int(mask_test.sum()), 16_670)
check(
    "protocol: test illicit count", int((txs.loc[mask_test, "class"] == 1).sum()), 1_083
)

## 5. Leakage-Safe Feature Normalization & Audit

### Feature Scaling
Neural networks (MLP and GraphSAGE) are sensitive to feature scales. We apply `StandardScaler`, observing strict leakage controls:
- **Selection model:** `StandardScaler` is fit exclusively on the `fit` window (steps 1–24).
- **Final refit model:** `StandardScaler` is fit exclusively on the full `train` window (steps 1–34).
- Validation and test features are transformed using parameters $(\mu, \sigma)$ calculated from the respective training splits. Test data is never seen by the scaler.

### Leakage Audit Table
Every feature category and processing step is audited against the 6 core leakage questions.

In [ ]:
audit_rows = [
    {
        "feature_group": "Local_feature_* (93)",
        "available_at_prediction_time": "yes",
        "uses_future_steps": "no",
        "uses_labels": "no",
        "uses_test_data": "no",
        "uses_full_graph": "no",
        "verdict": "ASSUMED-SAFE",
        "reason": "Official Elliptic++ local transaction features computed per time step.",
    },
    {
        "feature_group": "Aggregate_feature_* (72)",
        "available_at_prediction_time": "yes",
        "uses_future_steps": "no",
        "uses_labels": "no",
        "uses_test_data": "no",
        "uses_full_graph": "no",
        "verdict": "ASSUMED-SAFE",
        "reason": "Official Elliptic++ aggregate features computed within the transaction step.",
    },
    {
        "feature_group": "txs_edgelist (graph edges)",
        "available_at_prediction_time": "yes",
        "uses_future_steps": "no",
        "uses_labels": "no",
        "uses_test_data": "no",
        "uses_full_graph": "no",
        "verdict": "SAFE",
        "reason": "Intra-step transaction edges; zero cross-step temporal leakage verified.",
    },
    {
        "feature_group": "domain block (17)",
        "available_at_prediction_time": "yes",
        "uses_future_steps": "no",
        "uses_labels": "no",
        "uses_test_data": "no",
        "uses_full_graph": "no",
        "verdict": "EXCLUDED-REDUNDANT",
        "reason": "Collinear with Local features (|r| > 0.98); 16,405 missing values.",
    },
    {
        "feature_group": "txId, Time step",
        "available_at_prediction_time": "yes",
        "uses_future_steps": "no",
        "uses_labels": "no",
        "uses_test_data": "no",
        "uses_full_graph": "no",
        "verdict": "EXCLUDED-ID",
        "reason": "Identifiers and temporal partition variables; excluded from model input.",
    },
    {
        "feature_group": "Transductive graph stats (EDA)",
        "available_at_prediction_time": "no",
        "uses_future_steps": "yes",
        "uses_labels": "no",
        "uses_test_data": "yes",
        "uses_full_graph": "yes",
        "verdict": "EXCLUDED-LEAK",
        "reason": "EDA component/degree stats computed across all 49 steps are excluded.",
    },
    {
        "feature_group": "StandardScaler parameters",
        "available_at_prediction_time": "yes",
        "uses_future_steps": "no",
        "uses_labels": "no",
        "uses_test_data": "no",
        "uses_full_graph": "no",
        "verdict": "SAFE",
        "reason": "Fitted strictly on historical training nodes only (1-24 / 1-34).",
    },
]
audit_table = pd.DataFrame(audit_rows)
save_table("leakage_audit.csv", audit_table)
display(audit_table)

# Raw feature matrix for all 203,769 nodes
raw_features = txs[model_features].to_numpy(dtype="float32")

# 1. Scaler for Model Selection (Fit on 1-24 only)
scaler_selection = StandardScaler()
scaler_selection.fit(raw_features[mask_fit])
X_scaled_selection = scaler_selection.transform(raw_features).astype("float32")

# 2. Scaler for Final Refit (Fit on 1-34 only)
scaler_final = StandardScaler()
scaler_final.fit(raw_features[mask_train])
X_scaled_final = scaler_final.transform(raw_features).astype("float32")

# Target labels: 1 for illicit, 0 for licit, -1 for unknown
y_all = np.where(txs["class"] == 1, 1.0, np.where(txs["class"] == 2, 0.0, -1.0)).astype(
    "float32"
)

# Convert to PyTorch tensors
X_selection_tensor = torch.tensor(X_scaled_selection, dtype=torch.float32)
X_final_tensor = torch.tensor(X_scaled_final, dtype=torch.float32)
y_tensor = torch.tensor(y_all, dtype=torch.float32).unsqueeze(1)

check("leakage: scaler fit on historical data only", True, True)
check(
    "leakage: feature tensor shape matches node universe",
    tuple(X_selection_tensor.shape),
    (203_769, 165),
)
check(
    "leakage: labels tensor shape matches node universe",
    tuple(y_tensor.shape),
    (203_769, 1),
)

save_json(
    "feature_list.json",
    {
        "model_features": model_features,
        "n_features": len(model_features),
        "excluded_domain_columns": domain_columns,
    },
)

## 6. Model Architectures: MLP vs GraphSAGE

To directly answer whether graph neighborhood aggregation improves fraud detection over isolated feature representations, we evaluate two paired neural architectures with matching capacity:

1. **Feature-Only Neural Baseline (`MLP`):**
   - 2-layer Feedforward Dense Network
   - Input (165) -> Linear(165, 128) -> ReLU -> Dropout(0.3) -> Linear(128, 1) -> Logits
   - Processes each transaction independently without graph edges.

2. **Homogeneous Graph Neural Network (`GraphSAGE`):**
   - 2-layer GraphSAGE with Mean Aggregation:
     $$h_v^{(1)} = \text{ReLU}\left( W_1^{(1)} h_v^{(0)} + W_2^{(1)} \cdot \text{MEAN}_{u \in \mathcal{N}(v)} h_u^{(0)} \right)$$
     $$z_v = W_1^{(2)} h_v^{(1)} + W_2^{(2)} \cdot \text{MEAN}_{u \in \mathcal{N}(v)} h_u^{(1)}$$
   - Input (165) -> SAGEConv(165, 128) -> ReLU -> Dropout(0.3) -> SAGEConv(128, 1) -> Logits

Both architectures are implemented cleanly in PyTorch, with a built-in sparse mean aggregation fallback if PyG is not installed in the execution environment.

In [ ]:
class SAGEConvFallback(nn.Module):
    """Self-contained SAGEConv layer with mean aggregation using PyTorch sparse operations."""

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.lin_l = nn.Linear(in_channels, out_channels, bias=True)
        self.lin_r = nn.Linear(in_channels, out_channels, bias=False)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin_l.weight)
        nn.init.zeros_(self.lin_l.bias)
        nn.init.xavier_uniform_(self.lin_r.weight)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        num_nodes = x.size(0)
        src, dst = edge_index[0], edge_index[1]

        # Compute mean neighbor aggregation
        deg = (
            torch.bincount(dst, minlength=num_nodes).float().clamp(min=1.0).unsqueeze(1)
        )

        # Scatter sum neighbor features
        out = torch.zeros_like(x)
        out.index_add_(0, dst, x[src])
        out = out / deg

        return self.lin_l(x) + self.lin_r(out)


class GraphSAGENet(nn.Module):
    """2-Layer GraphSAGE model for homogeneous transaction graph."""

    def __init__(
        self,
        in_channels: int = 165,
        hidden_channels: int = 128,
        out_channels: int = 1,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.dropout = dropout
        if PYG_AVAILABLE:
            self.conv1 = SAGEConv(in_channels, hidden_channels, aggr="mean")
            self.conv2 = SAGEConv(hidden_channels, out_channels, aggr="mean")
        else:
            self.conv1 = SAGEConvFallback(in_channels, hidden_channels)
            self.conv2 = SAGEConvFallback(hidden_channels, out_channels)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        out = self.conv2(h, edge_index)
        return out


class MLPNet(nn.Module):
    """2-Layer MLP baseline with matching dimensions and dropout."""

    def __init__(
        self,
        in_channels: int = 165,
        hidden_channels: int = 128,
        out_channels: int = 1,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.dropout = dropout
        self.fc1 = nn.Linear(in_channels, hidden_channels)
        self.fc2 = nn.Linear(hidden_channels, out_channels)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor = None) -> torch.Tensor:
        h = self.fc1(x)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        out = self.fc2(h)
        return out


# Instantiate test models to verify parameter counts and feedforward
test_sage = GraphSAGENet(165, 128, 1, 0.3)
test_mlp = MLPNet(165, 128, 1, 0.3)

sage_params = sum(p.numel() for p in test_sage.parameters() if p.requires_grad)
mlp_params = sum(p.numel() for p in test_mlp.parameters() if p.requires_grad)

print(f"GraphSAGE trainable parameters: {sage_params:,}")
print(f"MLP Baseline trainable parameters: {mlp_params:,}")

check(
    "architecture: GraphSAGE output dimension is 1 logit",
    test_sage(X_selection_tensor[:10], edge_index[:, :20]).shape,
    (10, 1),
)
check(
    "architecture: MLP output dimension is 1 logit",
    test_mlp(X_selection_tensor[:10]).shape,
    (10, 1),
)

## 7. Training Pipeline with Validation Early Stopping

### Loss Formulation & Class Imbalance
Elliptic++ transaction labels have severe class imbalance (~1:7.6 illicit-to-licit in train 1–34; ~1:9.6 in fit 1–24).
We employ `nn.BCEWithLogitsLoss(pos_weight=pos_weight)`:
$$\text{pos\_weight} = \frac{N_{\text{licit}}}{N_{\text{illicit}}}$$
Calculated strictly on historical training labels (never using test labels).

### Early Stopping & Optimization
- **Optimizer:** Adam with learning rate $\eta = 0.005$, weight decay $10^{-4}$.
- **Metric:** Validation PR-AUC on historical steps 25–34 (matching the primary selection metric).
- **Patience:** 20 epochs (max 150 epochs).
- **Refit Strategy:** After model selection on the `fit` window (1–24), the final model is refit on the full historical `train` window (1–34) for the optimal epoch count.

In [ ]:
def compute_pos_weight(mask: np.ndarray) -> torch.Tensor:
    """Compute pos_weight = negative_samples / positive_samples strictly on the masked split."""
    labels = txs.loc[mask, "class"].values
    positives = np.sum(labels == 1)
    negatives = np.sum(labels == 2)
    weight = negatives / max(1, positives)
    return torch.tensor([weight], dtype=torch.float32)


pos_weight_fit = compute_pos_weight(mask_fit).to(device)
pos_weight_train = compute_pos_weight(mask_train).to(device)

print(f"Loss pos_weight (fit 1-24): {pos_weight_fit.item():.3f}")
print(f"Loss pos_weight (train 1-34): {pos_weight_train.item():.3f}")

check(
    "training: fit pos_weight calculated on fit data only",
    round(pos_weight_fit.item(), 2),
    9.64,
)
check(
    "training: train pos_weight calculated on train data only",
    round(pos_weight_train.item(), 2),
    7.63,
)


def train_model(
    model_cls,
    X_tensor: torch.Tensor,
    edge_idx: torch.Tensor,
    train_mask: np.ndarray,
    val_mask: np.ndarray,
    pos_weight: torch.Tensor,
    max_epochs: int = 150,
    patience: int = 20,
    lr: float = 0.005,
    weight_decay: float = 1e-4,
    seed: int = SEED,
) -> tuple:
    """Train a model with validation PR-AUC early stopping."""
    torch.manual_seed(seed)
    model = model_cls().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    x = X_tensor.to(device)
    edges = edge_idx.to(device)
    y_true_fit = y_tensor.to(device)

    train_indices = torch.tensor(
        np.where(train_mask)[0], dtype=torch.long, device=device
    )
    val_indices = torch.tensor(np.where(val_mask)[0], dtype=torch.long, device=device)
    y_val_np = txs.loc[val_mask, "class"].values == 1

    best_val_pr_auc = -1.0
    best_epoch = 0
    best_state = None
    epochs_no_improve = 0

    history = {"epoch": [], "train_loss": [], "val_pr_auc": [], "val_roc_auc": []}

    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(x, edges)
        loss = criterion(logits[train_indices], y_true_fit[train_indices])
        loss.backward()
        optimizer.step()

        # Evaluate on validation mask
        model.eval()
        with torch.no_grad():
            val_logits = model(x, edges)[val_indices].squeeze(1)
            val_probs = torch.sigmoid(val_logits).cpu().numpy()

        val_pr_auc = float(average_precision_score(y_val_np, val_probs))
        val_roc_auc = float(roc_auc_score(y_val_np, val_probs))

        history["epoch"].append(epoch)
        history["train_loss"].append(float(loss.item()))
        history["val_pr_auc"].append(val_pr_auc)
        history["val_roc_auc"].append(val_roc_auc)

        if val_pr_auc > best_val_pr_auc:
            best_val_pr_auc = val_pr_auc
            best_epoch = epoch
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_epoch, best_val_pr_auc, pd.DataFrame(history)


def refit_model(
    model_cls,
    X_tensor: torch.Tensor,
    edge_idx: torch.Tensor,
    train_mask: np.ndarray,
    pos_weight: torch.Tensor,
    epochs: int,
    lr: float = 0.005,
    weight_decay: float = 1e-4,
    seed: int = SEED,
) -> nn.Module:
    """Refit model on full historical training set (1-34) for fixed epoch count."""
    torch.manual_seed(seed)
    model = model_cls().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    x = X_tensor.to(device)
    edges = edge_idx.to(device)
    y_true = y_tensor.to(device)
    train_indices = torch.tensor(
        np.where(train_mask)[0], dtype=torch.long, device=device
    )

    model.train()
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        logits = model(x, edges)
        loss = criterion(logits[train_indices], y_true[train_indices])
        loss.backward()
        optimizer.step()

    return model

In [ ]:
print("--- 1. Training Feature-Only MLP Selection Model (Fit 1-24) ---")
start_time = time.time()
mlp_selection_model, mlp_best_epoch, mlp_val_pr_auc, mlp_history = train_model(
    MLPNet, X_selection_tensor, edge_index, mask_fit, mask_val, pos_weight_fit
)
mlp_fit_time = time.time() - start_time
print(
    f"MLP Selection: Best Epoch {mlp_best_epoch} | Validation PR-AUC {mlp_val_pr_auc:.4f} in {mlp_fit_time:.1f}s"
)

print("\n--- 2. Training GraphSAGE Selection Model (Fit 1-24) ---")
start_time = time.time()
sage_selection_model, sage_best_epoch, sage_val_pr_auc, sage_history = train_model(
    GraphSAGENet, X_selection_tensor, edge_index, mask_fit, mask_val, pos_weight_fit
)
sage_fit_time = time.time() - start_time
print(
    f"GraphSAGE Selection: Best Epoch {sage_best_epoch} | Validation PR-AUC {sage_val_pr_auc:.4f} in {sage_fit_time:.1f}s"
)

print("\n--- 3. Refitting Models on Full Train Period (1-34) ---")
mlp_final_model = refit_model(
    MLPNet,
    X_final_tensor,
    edge_index,
    mask_train,
    pos_weight_train,
    epochs=mlp_best_epoch,
)
sage_final_model = refit_model(
    GraphSAGENet,
    X_final_tensor,
    edge_index,
    mask_train,
    pos_weight_train,
    epochs=sage_best_epoch,
)
print("Refitting complete for both MLP and GraphSAGE.")

# Save PyTorch checkpoints
torch.save(mlp_final_model.state_dict(), OUT_DIR / "mlp_model.pt")
torch.save(sage_final_model.state_dict(), OUT_DIR / "graphsage_model.pt")

# Plot Learning Curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(
    mlp_history["epoch"],
    mlp_history["train_loss"],
    label="MLP Train Loss",
    color="#1f77b4",
)
axes[0].plot(
    sage_history["epoch"],
    sage_history["train_loss"],
    label="GraphSAGE Train Loss",
    color="#2ca02c",
)
axes[0].set(xlabel="Epoch", ylabel="Loss", title="Training Loss (Fit Window 1-24)")
axes[0].legend()

axes[1].plot(
    mlp_history["epoch"],
    mlp_history["val_pr_auc"],
    label=f"MLP Val PR-AUC (best={mlp_val_pr_auc:.4f})",
    color="#1f77b4",
)
axes[1].plot(
    sage_history["epoch"],
    sage_history["val_pr_auc"],
    label=f"GraphSAGE Val PR-AUC (best={sage_val_pr_auc:.4f})",
    color="#2ca02c",
)
axes[1].axvline(mlp_best_epoch, color="#1f77b4", linestyle="--", alpha=0.6)
axes[1].axvline(sage_best_epoch, color="#2ca02c", linestyle="--", alpha=0.6)
axes[1].set(xlabel="Epoch", ylabel="PR-AUC", title="Validation PR-AUC (Window 25-34)")
axes[1].legend()
save_fig("learning_curves.png")

## 8. Operating Threshold Selection

Consistent with Phase 2, classification thresholds are chosen by **maximizing F1 score on the validation window (25–34) predictions only**.
Thresholds are completely frozen before conducting any evaluation on the test period (35–49).

In [ ]:
def get_model_probabilities(
    model: nn.Module, X_tensor: torch.Tensor, edge_idx: torch.Tensor
) -> np.ndarray:
    """Compute predicted sigmoid probabilities for all nodes."""
    model.eval()
    with torch.no_grad():
        logits = model(X_tensor.to(device), edge_idx.to(device)).squeeze(1)
        probs = torch.sigmoid(logits).cpu().numpy()
    return probs


def select_threshold(y_true: np.ndarray, scores: np.ndarray) -> tuple:
    """Find F1-maximizing threshold on validation data only."""
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype="float64")
    best_threshold, best_f1 = 0.5, -1.0
    for candidate in np.round(np.linspace(0.005, 0.995, 199), 4):
        value = f1_score(y_true, (scores >= candidate).astype(int), zero_division=0)
        if value > best_f1:
            best_threshold, best_f1 = float(candidate), float(value)
    return best_threshold, best_f1


# Generate validation probabilities from selection models
val_y = (txs.loc[mask_val, "class"] == 1).values.astype(int)
mlp_val_probs = get_model_probabilities(
    mlp_selection_model, X_selection_tensor, edge_index
)[mask_val]
sage_val_probs = get_model_probabilities(
    sage_selection_model, X_selection_tensor, edge_index
)[mask_val]

mlp_thr, mlp_val_f1 = select_threshold(val_y, mlp_val_probs)
sage_thr, sage_val_f1 = select_threshold(val_y, sage_val_probs)

operating_points = {
    "mlp": {
        "threshold": mlp_thr,
        "validation_f1": mlp_val_f1,
        "best_epoch": mlp_best_epoch,
    },
    "graphsage": {
        "threshold": sage_thr,
        "validation_f1": sage_val_f1,
        "best_epoch": sage_best_epoch,
    },
}

print(
    f"MLP Operating Threshold       : {mlp_thr:.3f} (Validation F1: {mlp_val_f1:.4f})"
)
print(
    f"GraphSAGE Operating Threshold : {sage_thr:.3f} (Validation F1: {sage_val_f1:.4f})"
)

check(
    "threshold: MLP threshold selected on validation only",
    select_threshold(val_y, mlp_val_probs)[0],
    mlp_thr,
)
check(
    "threshold: GraphSAGE threshold selected on validation only",
    select_threshold(val_y, sage_val_probs)[0],
    sage_thr,
)

## 9. Primary Model Evaluation (Test Period 35–49)

We evaluate the models on the primary frozen test period (time steps 35–49; 16,670 labeled transactions; 1,083 illicit; 6.50% illicit prevalence).

We benchmark the new models against the complete progression of baselines:
1. **Prevalence Baseline:** Constant probability equal to test illicit prevalence.
2. **Logistic Regression:** Linear feature baseline (Phase 2).
3. **XGBoost Baseline:** Initial 500-tree tree-cap baseline (Phase 2).
4. **XGBoost Optimized:** Frozen 165-feature baseline (Phase 2: PR-AUC 0.8013, ROC-AUC 0.9281).
5. **MLP Baseline:** 2-layer feature-only neural network.
6. **GraphSAGE Baseline:** 2-layer homogeneous graph neural network.

In [ ]:
def evaluate(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    """Compute all standard evaluation metrics at a fixed operating threshold."""
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype="float64")
    predicted = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    prevalence = float(y_true.mean())
    pr_auc = float(average_precision_score(y_true, scores))
    return {
        "n": int(y_true.size),
        "illicit": int(y_true.sum()),
        "prevalence": prevalence,
        "threshold": float(threshold),
        "pr_auc": pr_auc,
        "pr_auc_lift_over_prevalence": pr_auc / prevalence
        if prevalence
        else float("nan"),
        "roc_auc": float(roc_auc_score(y_true, scores))
        if 0 < y_true.sum() < y_true.size
        else float("nan"),
        "precision": float(precision_score(y_true, predicted, zero_division=0)),
        "recall": float(recall_score(y_true, predicted, zero_division=0)),
        "f1": float(f1_score(y_true, predicted, zero_division=0)),
        "predicted_positives": int(predicted.sum()),
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }


# Generate full graph test predictions from final refit models
mlp_test_probs = get_model_probabilities(mlp_final_model, X_final_tensor, edge_index)[
    mask_test
]
sage_test_probs = get_model_probabilities(sage_final_model, X_final_tensor, edge_index)[
    mask_test
]
y_test = (txs.loc[mask_test, "class"] == 1).values.astype(int)
test_prevalence = float(y_test.mean())

# Register models
register_model(
    "mlp",
    "MLP baseline",
    mlp_final_model,
    {"val": mlp_val_probs, "test": mlp_test_probs},
)
register_model(
    "graphsage",
    "GraphSAGE baseline",
    sage_final_model,
    {"val": sage_val_probs, "test": sage_test_probs},
)

# Evaluate MLP and GraphSAGE
eval_mlp = evaluate(y_test, mlp_test_probs, operating_points["mlp"]["threshold"])
eval_sage = evaluate(
    y_test, sage_test_probs, operating_points["graphsage"]["threshold"]
)

# Include frozen reference baselines from Phase 2
eval_prevalence = {
    "n": len(y_test),
    "illicit": int(y_test.sum()),
    "prevalence": test_prevalence,
    "threshold": test_prevalence,
    "pr_auc": test_prevalence,
    "pr_auc_lift_over_prevalence": 1.0,
    "roc_auc": 0.5000,
    "precision": test_prevalence,
    "recall": 1.0000,
    "f1": 2 * test_prevalence / (1 + test_prevalence),
    "predicted_positives": len(y_test),
    "tp": int(y_test.sum()),
    "fp": int(len(y_test) - y_test.sum()),
    "tn": 0,
    "fn": 0,
}
eval_logistic = {
    "n": len(y_test),
    "illicit": int(y_test.sum()),
    "prevalence": test_prevalence,
    "threshold": 0.655,
    "pr_auc": 0.2917,
    "pr_auc_lift_over_prevalence": 0.2917 / test_prevalence,
    "roc_auc": 0.8828,
    "precision": 0.3547,
    "recall": 0.5734,
    "f1": 0.4385,
    "predicted_positives": 1751,
    "tp": 621,
    "fp": 1130,
    "tn": 14457,
    "fn": 462,
}
eval_xgb_base = {
    "n": len(y_test),
    "illicit": int(y_test.sum()),
    "prevalence": test_prevalence,
    "threshold": 0.515,
    "pr_auc": 0.8007,
    "pr_auc_lift_over_prevalence": 0.8007 / test_prevalence,
    "roc_auc": 0.9317,
    "precision": 0.8258,
    "recall": 0.7516,
    "f1": 0.7870,
    "predicted_positives": 986,
    "tp": 814,
    "fp": 172,
    "tn": 15415,
    "fn": 269,
}
eval_xgb_opt = {
    "n": len(y_test),
    "illicit": int(y_test.sum()),
    "prevalence": test_prevalence,
    "threshold": 0.435,
    "pr_auc": 0.8013,
    "pr_auc_lift_over_prevalence": 0.8013 / test_prevalence,
    "roc_auc": 0.9281,
    "precision": 0.7960,
    "recall": 0.7682,
    "f1": 0.7818,
    "predicted_positives": 1045,
    "tp": 832,
    "fp": 213,
    "tn": 15374,
    "fn": 251,
}

test_evaluations = {
    "Prevalence Baseline": eval_prevalence,
    "Logistic Regression": eval_logistic,
    "XGBoost Baseline": eval_xgb_base,
    "XGBoost Optimized (Frozen)": eval_xgb_opt,
    "MLP Baseline": eval_mlp,
    "GraphSAGE Baseline": eval_sage,
}

print("=== Primary Model Evaluation on Test Period (Steps 35-49) ===")
summary_df = pd.DataFrame(test_evaluations).T[
    ["pr_auc", "roc_auc", "precision", "recall", "f1", "tp", "fp", "fn", "threshold"]
]
display(summary_df.round(4))

# Plot Precision-Recall & ROC Curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# PR Curve
p_mlp, r_mlp, _ = precision_recall_curve(y_test, mlp_test_probs)
p_sage, r_sage, _ = precision_recall_curve(y_test, sage_test_probs)
axes[0].plot(
    r_sage,
    p_sage,
    label=f"GraphSAGE (PR-AUC={eval_sage['pr_auc']:.4f})",
    color="#2ca02c",
    lw=2,
)
axes[0].plot(
    r_mlp,
    p_mlp,
    label=f"MLP (PR-AUC={eval_mlp['pr_auc']:.4f})",
    color="#1f77b4",
    lw=1.5,
)
axes[0].axhline(
    0.8013, color="#d62728", linestyle=":", label="XGBoost Opt Baseline (0.8013)"
)
axes[0].axhline(
    test_prevalence,
    color="grey",
    linestyle="--",
    label=f"Prevalence ({test_prevalence:.4f})",
)
axes[0].set(
    xlabel="Recall", ylabel="Precision", title="Precision-Recall Curve (Test 35-49)"
)
axes[0].legend(loc="upper right", fontsize=8)

# ROC Curve
for name, probs, color in [
    ("GraphSAGE", sage_test_probs, "#2ca02c"),
    ("MLP", mlp_test_probs, "#1f77b4"),
]:
    order = np.argsort(probs)
    y_sorted = y_test[order]
    tpr = np.cumsum(y_sorted[::-1]) / y_sorted.sum()
    fpr = np.cumsum((1 - y_sorted)[::-1]) / (1 - y_sorted).sum()
    auc = average_precision_score(y_test, probs)
    axes[1].plot(fpr, tpr, label=f"{name}", color=color, lw=1.8)
axes[1].plot([0, 1], [0, 1], color="grey", linestyle=":")
axes[1].set(
    xlabel="False Positive Rate",
    ylabel="True Positive Rate",
    title="ROC Curve (Test 35-49)",
)
axes[1].legend(loc="lower right", fontsize=8)
save_fig("test_pr_curve_and_roc.png")

check(
    "evaluation: confusion matrix arithmetic closes for GraphSAGE",
    eval_sage["tp"] + eval_sage["fp"] + eval_sage["tn"] + eval_sage["fn"],
    len(y_test),
)
check(
    "evaluation: confusion matrix arithmetic closes for MLP",
    eval_mlp["tp"] + eval_mlp["fp"] + eval_mlp["tn"] + eval_mlp["fn"],
    len(y_test),
)

## 10. Temporal Evaluation & Sub-Window Breakdown

Evaluating models strictly across temporal sub-windows reveals whether predictive performance is robust or degrades over time:

1. **Early Test Window (`35–42`):** 9,983 labeled transactions, 914 illicit (prevalence = 9.16%). High-activity regime.
2. **Recent Drift Window (`43–49`):** 6,687 labeled transactions, 169 illicit (prevalence = 2.53%). Low-prevalence regime.
3. **Full Test Window (`35–49`):** Primary 15-step benchmark.

In [ ]:
test_step_vals = txs.loc[mask_test, "Time step"].values

temporal_subwindows = {
    "test_full": ("35-49 (Primary)", np.ones(len(y_test), dtype=bool)),
    "test_early": ("35-42 (Early Test)", test_step_vals <= TEST_EARLY_MAX),
    "test_drift": ("43-49 (Recent Drift)", test_step_vals >= DRIFT_MIN),
}

temporal_rows = []
for win_key, (win_label, win_mask) in temporal_subwindows.items():
    y_win = y_test[win_mask]
    n_win = len(y_win)
    ill_win = int(y_win.sum())
    prev_win = float(y_win.mean())

    # MLP metrics
    mlp_win_eval = evaluate(
        y_win, mlp_test_probs[win_mask], operating_points["mlp"]["threshold"]
    )
    # GraphSAGE metrics
    sage_win_eval = evaluate(
        y_win, sage_test_probs[win_mask], operating_points["graphsage"]["threshold"]
    )

    # Reference XGBoost metrics for comparison
    xgb_pr_auc = (
        0.8013
        if win_key == "test_full"
        else (0.9215 if win_key == "test_early" else 0.0427)
    )

    temporal_rows.append(
        {
            "window": win_label,
            "steps": {
                "test_full": "35-49",
                "test_early": "35-42",
                "test_drift": "43-49",
            }[win_key],
            "labeled": n_win,
            "illicit": ill_win,
            "prevalence": prev_win,
            "xgboost_pr_auc": xgb_pr_auc,
            "mlp_pr_auc": mlp_win_eval["pr_auc"],
            "mlp_f1": mlp_win_eval["f1"],
            "graphsage_pr_auc": sage_win_eval["pr_auc"],
            "graphsage_f1": sage_win_eval["f1"],
            "graphsage_tp": sage_win_eval["tp"],
            "graphsage_fp": sage_win_eval["fp"],
            "graphsage_fn": sage_win_eval["fn"],
        }
    )

temporal_table = pd.DataFrame(temporal_rows)
save_table("temporal_metrics.csv", temporal_table)
display(temporal_table.round(4))

# Plot Temporal PR-AUC Comparison
fig, ax = plt.subplots(figsize=(8.5, 4.5))
bar_width = 0.25
indices = np.arange(len(temporal_table))

ax.bar(
    indices - bar_width,
    temporal_table["xgboost_pr_auc"],
    bar_width,
    label="XGBoost Optimized",
    color="#d62728",
)
ax.bar(
    indices,
    temporal_table["mlp_pr_auc"],
    bar_width,
    label="MLP Baseline",
    color="#1f77b4",
)
ax.bar(
    indices + bar_width,
    temporal_table["graphsage_pr_auc"],
    bar_width,
    label="GraphSAGE Baseline",
    color="#2ca02c",
)

for idx, prev in enumerate(temporal_table["prevalence"]):
    ax.plot(
        [idx - 0.35, idx + 0.35],
        [prev, prev],
        color="grey",
        linestyle="--",
        linewidth=1.2,
    )

ax.set_xticks(indices)
ax.set_xticklabels(temporal_table["steps"])
ax.set(
    xlabel="Time Step Sub-window",
    ylabel="PR-AUC",
    title="PR-AUC by Temporal Window (Dashed = Prevalence)",
)
ax.legend(fontsize=9)
save_fig("temporal_pr_auc.png")

check(
    "temporal: sub-windows partition labeled count",
    int(temporal_table.loc[temporal_table["steps"] != "35-49", "labeled"].sum()),
    int(temporal_table.loc[temporal_table["steps"] == "35-49", "labeled"].iloc[0]),
)

## 11. Controlled Ablation: MLP vs GraphSAGE (Isolated vs Connected Nodes)

To understand *where* graph message passing provides value, we decompose test nodes into:
1. **Isolated Nodes (`degree == 0`):** Transactions with zero transaction-to-transaction edges in `txs_edgelist.csv` (143,084 dataset-wide). GraphSAGE receives zero neighbor messages and reduces to a self-node transformation.
2. **Connected Nodes (`degree >= 1`):** Transactions with at least one transaction edge (60,685 dataset-wide). GraphSAGE aggregates features from adjacent transactions.

In [ ]:
test_degrees = txs.loc[mask_test, "total_degree"].values

node_groups = {
    "all_test_nodes": ("All Test Nodes", np.ones(len(y_test), dtype=bool)),
    "connected_nodes": ("Connected Nodes (Degree >= 1)", test_degrees >= 1),
    "isolated_nodes": ("Isolated Nodes (Degree == 0)", test_degrees == 0),
}

ablation_rows = []
for grp_label, grp_mask in node_groups.values():
    y_grp = y_test[grp_mask]
    n_grp = len(y_grp)
    ill_grp = int(y_grp.sum())
    prev_grp = float(y_grp.mean())

    mlp_grp_eval = evaluate(
        y_grp, mlp_test_probs[grp_mask], operating_points["mlp"]["threshold"]
    )
    sage_grp_eval = evaluate(
        y_grp, sage_test_probs[grp_mask], operating_points["graphsage"]["threshold"]
    )

    ablation_rows.append(
        {
            "group": grp_label,
            "n_nodes": n_grp,
            "illicit": ill_grp,
            "prevalence": prev_grp,
            "mlp_pr_auc": mlp_grp_eval["pr_auc"],
            "mlp_f1": mlp_grp_eval["f1"],
            "graphsage_pr_auc": sage_grp_eval["pr_auc"],
            "graphsage_f1": sage_grp_eval["f1"],
            "pr_auc_delta_sage_vs_mlp": sage_grp_eval["pr_auc"]
            - mlp_grp_eval["pr_auc"],
            "f1_delta_sage_vs_mlp": sage_grp_eval["f1"] - mlp_grp_eval["f1"],
        }
    )

ablation_table = pd.DataFrame(ablation_rows)
save_table("ablation_results.csv", ablation_table)
display(ablation_table.round(4))

# Plot Connected vs Isolated Performance
fig, ax = plt.subplots(figsize=(8, 4.5))
bar_width = 0.35
grp_indices = np.arange(len(ablation_table))

ax.bar(
    grp_indices - bar_width / 2,
    ablation_table["mlp_pr_auc"],
    bar_width,
    label="MLP (Features Only)",
    color="#1f77b4",
)
ax.bar(
    grp_indices + bar_width / 2,
    ablation_table["graphsage_pr_auc"],
    bar_width,
    label="GraphSAGE (Features + Graph)",
    color="#2ca02c",
)
ax.set_xticks(grp_indices)
ax.set_xticklabels([r["group"] for r in ablation_rows], fontsize=9)
ax.set(ylabel="PR-AUC", title="Ablation: MLP vs GraphSAGE by Neighborhood Connectivity")
ax.legend()
save_fig("degree_vs_performance.png")

## 12. Error Analysis & Neighborhood Diagnostics

We dissect the failure modes of the GraphSAGE baseline:
1. **Temporal distribution of errors:** False Negatives (FN) and False Positives (FP) per time step.
2. **Degree vs Error Rate:** Does GraphSAGE error distribution correlate with transaction degree?
3. **Score Distributions:** Examining confidence separation between licit and illicit classes.

In [ ]:
sage_thr = operating_points["graphsage"]["threshold"]
sage_pred = (sage_test_probs >= sage_thr).astype(int)

error_df = pd.DataFrame(
    {
        "txId": txs.loc[mask_test, "txId"].values,
        "time_step": test_step_vals,
        "degree": test_degrees,
        "in_degree": txs.loc[mask_test, "in_degree"].values,
        "out_degree": txs.loc[mask_test, "out_degree"].values,
        "y_true": y_test,
        "score": sage_test_probs,
        "predicted": sage_pred,
    }
)

error_df["error_type"] = np.select(
    [
        (error_df.y_true == 1) & (error_df.predicted == 0),
        (error_df.y_true == 0) & (error_df.predicted == 1),
    ],
    ["FN", "FP"],
    default="correct",
)

# 1. Errors by time step
per_step_errors = (
    error_df.groupby("time_step")
    .apply(
        lambda b: pd.Series(
            {
                "n": len(b),
                "illicit": int(b.y_true.sum()),
                "tp": int(((b.y_true == 1) & (b.predicted == 1)).sum()),
                "fp": int(((b.y_true == 0) & (b.predicted == 1)).sum()),
                "fn": int(((b.y_true == 1) & (b.predicted == 0)).sum()),
                "tn": int(((b.y_true == 0) & (b.predicted == 0)).sum()),
                "precision": float(
                    precision_score(b.y_true, b.predicted, zero_division=0)
                ),
                "recall": float(recall_score(b.y_true, b.predicted, zero_division=0)),
                "f1": float(f1_score(b.y_true, b.predicted, zero_division=0)),
            }
        )
    )
    .reset_index()
)
save_table("errors_by_step.csv", per_step_errors)

# 2. Errors by degree bucket
error_df["degree_bucket"] = pd.cut(
    error_df["degree"],
    bins=[-1, 0, 1, 3, 10, 1000],
    labels=["0", "1", "2-3", "4-10", "11+"],
)
per_degree_errors = (
    error_df.groupby("degree_bucket", observed=False)
    .apply(
        lambda b: pd.Series(
            {
                "n": len(b),
                "illicit": int(b.y_true.sum()),
                "tp": int(((b.y_true == 1) & (b.predicted == 1)).sum()),
                "fp": int(((b.y_true == 0) & (b.predicted == 1)).sum()),
                "fn": int(((b.y_true == 1) & (b.predicted == 0)).sum()),
                "precision": float(
                    precision_score(b.y_true, b.predicted, zero_division=0)
                ),
                "recall": float(recall_score(b.y_true, b.predicted, zero_division=0)),
                "f1": float(f1_score(b.y_true, b.predicted, zero_division=0)),
            }
        )
    )
    .reset_index()
)
save_table("errors_by_degree.csv", per_degree_errors)

print("Errors by Node Degree Bucket:")
display(per_degree_errors.round(4))

# Plot Error Analysis
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
axes[0].plot(
    per_step_errors["time_step"],
    per_step_errors["fn"],
    marker="o",
    label="False Negatives (FN)",
    color="#d62728",
)
axes[0].plot(
    per_step_errors["time_step"],
    per_step_errors["fp"],
    marker="s",
    label="False Positives (FP)",
    color="#ff7f0e",
)
axes[0].set(
    xlabel="Time Step",
    ylabel="Count",
    title="GraphSAGE Errors per Time Step (Test 35-49)",
)
axes[0].legend()

axes[1].hist(
    error_df.loc[error_df.y_true == 1, "score"],
    bins=50,
    alpha=0.6,
    label="Illicit",
    color="#d62728",
)
axes[1].hist(
    error_df.loc[error_df.y_true == 0, "score"],
    bins=50,
    alpha=0.6,
    label="Licit",
    color="#1f77b4",
)
axes[1].axvline(
    sage_thr, color="black", linestyle="--", label=f"Threshold ({sage_thr:.3f})"
)
axes[1].set(
    xlabel="Predicted Score",
    ylabel="Count",
    title="GraphSAGE Predicted Probability Distribution",
    yscale="log",
)
axes[1].legend()
save_fig("error_analysis.png")

## 13. Model Comparison, Artifact Export & Automated Checks

Compiles all results, writes predictions, metadata JSONs, digest file, and executes the suite of automated sanity checks.

In [ ]:
# 1. Model Comparison Table
comp_rows = []
for model_name, eval_dict in test_evaluations.items():
    comp_rows.append(
        {
            "model": model_name,
            "pr_auc": eval_dict["pr_auc"],
            "roc_auc": eval_dict["roc_auc"],
            "precision": eval_dict["precision"],
            "recall": eval_dict["recall"],
            "f1": eval_dict["f1"],
            "tp": eval_dict["tp"],
            "fp": eval_dict["fp"],
            "fn": eval_dict["fn"],
            "threshold": eval_dict["threshold"],
        }
    )
model_comparison_table = pd.DataFrame(comp_rows)
save_table("model_comparison.csv", model_comparison_table)
display(model_comparison_table.round(4))

# 2. Export Predictions CSV
predictions_df = pd.DataFrame(
    {
        "txId": txs.loc[mask_test, "txId"].values,
        "time_step": test_step_vals,
        "degree": test_degrees,
        "y_true": y_test,
        "mlp_score": mlp_test_probs,
        "mlp_predicted": (
            mlp_test_probs >= operating_points["mlp"]["threshold"]
        ).astype(int),
        "graphsage_score": sage_test_probs,
        "graphsage_predicted": (
            sage_test_probs >= operating_points["graphsage"]["threshold"]
        ).astype(int),
    }
)
save_table("predictions.csv", predictions_df)

# 3. Export Hyperparameters JSON
hyperparameters_payload = {
    "graphsage": {
        "architecture": "2-layer SAGEConv (mean aggregation)",
        "in_channels": 165,
        "hidden_channels": 128,
        "out_channels": 1,
        "dropout": 0.3,
        "optimizer": "Adam",
        "lr": 0.005,
        "weight_decay": 0.0001,
        "loss": "BCEWithLogitsLoss",
        "pos_weight_fit": round(pos_weight_fit.item(), 4),
        "pos_weight_train": round(pos_weight_train.item(), 4),
        "best_epoch": sage_best_epoch,
        "validation_pr_auc": sage_val_pr_auc,
        "threshold": sage_thr,
    },
    "mlp": {
        "architecture": "2-layer Linear",
        "in_channels": 165,
        "hidden_channels": 128,
        "out_channels": 1,
        "dropout": 0.3,
        "best_epoch": mlp_best_epoch,
        "validation_pr_auc": mlp_val_pr_auc,
        "threshold": mlp_thr,
    },
    "seed": SEED,
    "device": str(device),
}
save_json("graphsage_hyperparameters.json", hyperparameters_payload)

# 4. Export Metrics JSON
metrics_payload = {
    "protocol": {
        "fit": "1-24",
        "validation": "25-34",
        "train": "1-34",
        "test": "35-49",
    },
    "evaluations": test_evaluations,
    "ablation": ablation_table.to_dict(orient="records"),
    "temporal": temporal_table.to_dict(orient="records"),
    "hyperparameters": hyperparameters_payload,
}
save_json("metrics.json", metrics_payload)

# 5. Digest Text
digest = f"""
BitcoinGraphGuard - Phase 3 GraphSAGE Baseline Digest ({time.strftime("%Y-%m-%d %H:%M")})
--------------------------------------------------------------------------------------
Graph Structure : 203,769 nodes, 234,355 directed edges (100% intra-step)
Features        : 165 transaction features (StandardScaler fit on training window only)
Population      : class 1 (illicit) + class 2 (licit); class 3 (unknown) excluded from loss
Protocol        : fit 1-24 | validation 25-34 | train 1-34 | test 35-49

Test 35-49 Results:
- Prevalence Baseline : PR-AUC {eval_prevalence["pr_auc"]:.4f} | ROC-AUC 0.5000
- Logistic Regression : PR-AUC {eval_logistic["pr_auc"]:.4f} | ROC-AUC {eval_logistic["roc_auc"]:.4f} | F1 {eval_logistic["f1"]:.4f}
- XGBoost Baseline    : PR-AUC {eval_xgb_base["pr_auc"]:.4f} | ROC-AUC {eval_xgb_base["roc_auc"]:.4f} | F1 {eval_xgb_base["f1"]:.4f}
- XGBoost Optimized   : PR-AUC {eval_xgb_opt["pr_auc"]:.4f} | ROC-AUC {eval_xgb_opt["roc_auc"]:.4f} | F1 {eval_xgb_opt["f1"]:.4f}
- MLP Baseline        : PR-AUC {eval_mlp["pr_auc"]:.4f} | ROC-AUC {eval_mlp["roc_auc"]:.4f} | F1 {eval_mlp["f1"]:.4f}
- GraphSAGE Baseline  : PR-AUC {eval_sage["pr_auc"]:.4f} | ROC-AUC {eval_sage["roc_auc"]:.4f} | F1 {eval_sage["f1"]:.4f}

Temporal PR-AUC (GraphSAGE vs XGBoost):
- Window 35-42 (Early): GraphSAGE {temporal_table.loc[temporal_table["steps"] == "35-42", "graphsage_pr_auc"].iloc[0]:.4f} vs XGBoost 0.9215
- Window 43-49 (Drift): GraphSAGE {temporal_table.loc[temporal_table["steps"] == "43-49", "graphsage_pr_auc"].iloc[0]:.4f} vs XGBoost 0.0427

Ablation (GraphSAGE vs MLP):
- Connected Nodes (deg >= 1): GraphSAGE PR-AUC {ablation_table.loc[ablation_table["group"].str.startswith("Connected"), "graphsage_pr_auc"].iloc[0]:.4f} vs MLP {ablation_table.loc[ablation_table["group"].str.startswith("Connected"), "mlp_pr_auc"].iloc[0]:.4f}
- Isolated Nodes (deg == 0) : GraphSAGE PR-AUC {ablation_table.loc[ablation_table["group"].str.startswith("Isolated"), "graphsage_pr_auc"].iloc[0]:.4f} vs MLP {ablation_table.loc[ablation_table["group"].str.startswith("Isolated"), "mlp_pr_auc"].iloc[0]:.4f}
"""
print(digest)
(OUT_DIR / "graphsage_digest.txt").write_text(digest, encoding="utf-8")

# Record and save sanity checks
check(
    "predictions: GraphSAGE prediction matches score >= threshold",
    int(
        (
            predictions_df["graphsage_predicted"]
            != (predictions_df["graphsage_score"] >= sage_thr).astype(int)
        ).sum()
    ),
    0,
)
check(
    "predictions: MLP prediction matches score >= threshold",
    int(
        (
            predictions_df["mlp_predicted"]
            != (predictions_df["mlp_score"] >= mlp_thr).astype(int)
        ).sum()
    ),
    0,
)

save_table("checks.csv", pd.DataFrame(CHECKS))

failures = [row for row in CHECKS if not row["ok"]]
print(f"\nSanity Checks Summary: {len(CHECKS)} total checks, {len(failures)} failures.")
if failures:
    display(pd.DataFrame(failures))
print(
    "Exported artifacts in output directory:", sorted(p.name for p in OUT_DIR.iterdir())
)

## 14. Conclusions & Next Steps

### Research Findings from Phase 3:
1. **Can homogeneous GraphSAGE beat the XGBoost baseline?**
   - The primary benchmark remains the frozen XGBoost baseline (**Test 35–49 PR-AUC: 0.8013**).
   - GraphSAGE operates purely on the 234,355 transaction-to-transaction edges.
2. **Structural Limits of Transaction-Only Graphs:**
   - 100% of transaction-to-transaction edges are confined within single time steps (zero cross-step temporal edges).
   - 70.2% of transactions (143,084 nodes) are isolated singletons in the transaction-only graph.
   - For isolated nodes, homogeneous message passing provides no neighbor context and reduces to individual node transformation.
3. **Implications for Phase 4 (Heterogeneous GNNs — RGCN / HGT):**
   - In Bitcoin, money flow and actor identity span multiple time steps through **wallet/address nodes** (`AddrTx`, `TxAddr`, `AddrAddr`).
   - Heterogeneous graph modeling is necessary to bridge time steps and connect the 70% of transactions that lack direct transaction-to-transaction links.

---
**STOP CONDITION:** Phase 3 is complete. Do NOT begin Phase 4 (RGCN / HGT) until Phase 3 artifacts are reviewed and signed off.